# 🌌 Unified Latents: Persistent Cloud Training Setup

This notebook provides a **robust, persistent** environment for training the Unified Latents framework. 
- 💾 **Persistence:** Your checkpoints are automatically saved to Google Drive.
- 🚀 **GPU Acceleration:** Optimized for T4/A100 GPUs.
- 🔬 **Extension Support:** Includes the 'Learned Noise Schedule' research extension.

### 1. 💾 Mount Google Drive
Mount your drive to keep your training checkpoints safe across sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_BASE = "/content/drive/MyDrive/UnifiedLatents_Outputs"
os.makedirs(DRIVE_BASE, exist_ok=True)
print(f"--- Drive Persistent Folder: {DRIVE_BASE} ---")

### 2. 🔄 Clone/Update Codebase
Enter your GitHub Personal Access Token (PAT) below to clone the repository securely.

In [ ]:
#@title Repository Setup { display-mode: "form" }
GITHUB_TOKEN = "YOUR_GITHUB_TOKEN" #@param {type:"string"}

if GITHUB_TOKEN == "YOUR_GITHUB_TOKEN":
    print("❌ ERROR: Please enter your GitHub Personal Access Token!")
else:
    %cd /content
    !rm -rf UnifiedLatents
    REPO_URL = f"https://{GITHUB_TOKEN}@github.com/GiGiKoneti/UnifiedLatents.git"
    !git clone {REPO_URL}
    %cd UnifiedLatents

    # --- SETUP PERSISTENCE SYMLINK ---
    !rm -rf outputs
    !ln -s {DRIVE_BASE} outputs
    print("--- Project Ready and Linked to Drive! ---")

### 3. 📦 Install Dependencies

In [ ]:
!pip install einops diffusers matplotlib PyYAML tqdm

### 4. 🚀 Stage 1: Joint Training (Learned Noise Schedule Extension)
This trains the Encoder, Prior, and Decoder together using the **Learned Noise Schedule** research extension.

In [ ]:
# Start Stage 1 with the research extension enabled
!python3 train.py --stage 1 --use-learned-schedule

### 5. 🔬 Research Analysis: Learned Noise Distribution
Once Stage 1 finishes (or hits a few dozen epochs), run this to see how the model learned the noise schedule.

In [ ]:
# Find your latest checkpoint
latest_ckpt = !ls -t outputs/ckpt_stage1_epoch*.pt | head -n 1
if latest_ckpt:
    print(f"--- Analyzing: {latest_ckpt[0]} ---")
    !python3 analyze_extension.py --ckpt {latest_ckpt[0]}
else:
    print("--- No checkpoints found yet! ---")

### 6. 🎨 Stage 2: Prior Refinement
Finalize your results with weighted prior refinement to produce high-fidelity reconstructions.

In [ ]:
!python3 train.py --stage 2 --use-learned-schedule --ckpt outputs/ckpt_stage1_final.pt